# Chart gallery

Every kind of chart pymetrix draws, with the code that makes it and the matplotlib it comes down to. Run the cells in
order; each draws a figure.

**How pymetrix figures work.** A figure is data: panels of layers (lines, bars, bands...), each holding its numbers.
pymetrix draws it with matplotlib's object-oriented interface, so:
- **any figure can be styled** in matplotlib's terms (`fig.line("gdp", color="C3")`, `fig.set(ylabel="%")`), each
  call recorded so the figure replays exactly;
- **any figure can be written out as matplotlib**: `fig.matplotlib_code()` is a standalone script, NumPy, pandas and
  matplotlib only, that draws the same chart from the same numbers;
- **in the app**, every figure has **Edit…**, **Save…** and **Copy matplotlib code**, and **Data → Plot series…**
  makes any of the series charts below.

In [ ]:
from pymetrix import (VAR, Dataset, Equation, coefficient_figure, figure_of, heatmap_figure, house_style,
                      set_house_style, surface_figure)
from pymetrix.figures import Figure, Line3d, Panel, vintages_figure

data = Dataset.example("sample_macro").with_series(gy="400*dlog(y)", gc="400*dlog(c)", gi="400*dlog(i)", gg="400*dlog(g)",
                                                   dc="d(c)", di="d(i)", dg="d(g)", dy="d(y)")
data

## Lines

`data.plot(names)` draws lines: `Axes.plot`, one call per series. A series on its own scale goes on a right-hand
axis (`right=`, matplotlib's `twinx`); `transform=` plots growth, year-on-year changes, an index or the log;
`line_labels=True` names each line at its end (`annotate`) instead of in a legend.

In [ ]:
fig_lines = data.plot(["gy", "gc"], span=("2015Q1", None), title="Growth, % a year", line_labels=True)
fig_lines

In [ ]:
fig_right = data.plot(["y", "i"], right="i", span=("2010Q1", None), title="Output and investment (right)")
fig_right

In [ ]:
fig_step = data.plot("y", kind="step", transform="yoy", span=("2018Q1", None), title="Year-on-year growth, as steps")
fig_step

## Bars

`kind="bar"` puts series side by side (`Axes.bar`, offset); `stacked=True` stacks them, negative values below zero
(`bottom=`); `kind="barh"` draws them across (`Axes.barh`); `bar_labels=True` writes each value (`bar_label`).

In [ ]:
fig_bars = data.plot(["gc", "gi"], kind="bar", span=("2023Q1", None), bar_labels=True, title="Side by side")
fig_bars

In [ ]:
fig_stacked = data.plot(["dc", "di", "dg"], kind="bar", stacked=True, span=("2021Q1", None), title="Stacked: negatives below zero")
fig_stacked

In [ ]:
fig_across = data.plot(["c", "i", "g"], kind="barh", span=("2024Q3", None), title="Across")
fig_across

## Areas and bands

`kind="area"` fills from zero (`fill_between`), stacked with `stacked=True`. Results draw bands the same way:
confidence intervals around a forecast or an impulse response.

In [ ]:
fig_area = data.plot(["c", "i", "g"], kind="area", stacked=True, span=("2010Q1", None), title="Stacked areas")
fig_area

## Shading, events and reference lines

`shade=` shades periods (`axvspan`): dates, a 0/1 series, or `"nber"` / `"cepr"` for the committees' recessions.
`events=` marks dates with a labelled dashed line (`axvline` and `text`); `reference=` marks values (`axhline`).

In [ ]:
fig_marks = data.plot("gy", shade="2008Q3:2009Q2=GFC, 2020Q1:2020Q2=COVID", events="2016Q2=Brexit vote",
                      reference="2=trend", span=("2005Q1", None), title="Growth, with recessions and events")
fig_marks

## Distributions

- `kind="histogram"`: a histogram per series with the normal curve (`hist`, `plot`); `bins=` sets the bins.
- `kind="ecdf"`: each series' empirical distribution, in one panel (`ecdf`): no bins to choose.
- `kind="box"`: box plots (`boxplot`).
- `kind="scatter"`: y against x, x first, with the least-squares line (`scatter`, `plot`).
- `kind="hist2d"`: a two-dimensional histogram (`hist2d`), for when points would hide each other.
- `kind="violin"`: violin plots (`violinplot`): each series' density, mirrored, with its median.
- `kind="hexbin"`: hexagonal bins (`hexbin`), for many observations; `bins=` sets the hexagons across.

In [ ]:
fig_hist = data.plot("gy", kind="histogram", bins=20, title="GDP growth")
fig_hist

In [ ]:
fig_ecdf = data.plot(["gy", "gc", "gi"], kind="ecdf", title="Empirical distributions")
fig_ecdf

In [ ]:
fig_box = data.plot(["gy", "gc", "gi"], kind="box", title="Box plots")
fig_box

In [ ]:
fig_scatter = data.plot(["gy", "gc"], kind="scatter", title="Consumption against output growth")
fig_scatter

In [ ]:
fig_hist2d = data.plot(["gy", "gc"], kind="hist2d", bins=12, title="The same, as counts")
fig_hist2d

In [ ]:
fig_violin = data.plot(["gy", "gc", "gi"], kind="violin", title="Violin plots")
fig_violin

In [ ]:
fig_hexbin = data.plot(["gy", "gc"], kind="hexbin", bins=15, title="The same, in hexagons")
fig_hexbin

## Macro charts

- **Contributions** (`kind="contributions"`): stacked parts with the total as a line; whatever the parts leave
  unexplained is drawn as *other*, so the bars always add up to the line.
- **Fans** (`kind="fan"`): a central path with bands at each coverage, darker towards the centre, and a dashed line
  where the forecast begins. The bands come from `{name}_lo90`/`{name}_hi90` or `{name}_se` columns; a forecast
  round's fan chart is one too.
- **Vintages** (`kind="vintages"`): releases of the same data, lighter to darker, the latest thickest.

In [ ]:
fig_contributions = data.plot(["dc", "di", "dg"], kind="contributions", total="dy", span=("2022Q1", None),
                              title="Contributions to the change in output")
fig_contributions

In [ ]:
import numpy as np

frame = data.frame[["gy"]].copy()
frame["gy_se"] = np.nan
frame.loc[frame.index[-8:], "gy_se"] = np.linspace(0.8, 3.0, 8)  # a forecast's standard errors, widening
fig_fan = Dataset.from_frame(frame).plot("gy", kind="fan", span=("2019Q1", None), title="A fan chart")
fig_fan

In [ ]:
releases = {f"release {k + 1}": data["gy"].iloc[-12:] + 0.3 * k for k in range(4)}  # each release revised up
fig_vintages = vintages_figure(releases, title="Revisions")
fig_vintages

## Surfaces and contours

A table over a grid (its columns along x, its rows along y, each value a height) is drawn by `surface_figure`:
- `kind="surface"` (`plot_surface`, on a 3-D panel, with a colour bar) or `"wireframe"` (`plot_wireframe`);
  `.view_init(elev=, azim=)` turns it, recorded like any style call;
- `kind="contourf"` (`contourf`, filled) or `"contour"` (`contour`, lines); `levels=`, and `clabel=True` writes
  each level on its line.

Dates work on any axis: a yield curve over time is a table of dates by maturities. `data.plot(names,
kind="surface")` draws series side by side the same way, and `Line3d` and `Scatter3d` draw lines and points in 3-D.
Results use them too: a volatility surface's Surface tab, a TVP-VAR's responses over time.

In [ ]:
import pandas as pd

maturities = pd.Index([0.25, 0.5, 1, 2, 3, 5, 7, 10, 20, 30], name="maturity (years)")
dates = data.frame.index[-40:].rename("date")
level = 2 + np.linspace(0, 1.5, 40) + 0.3 * np.sin(np.arange(40) / 4)  # Nelson-Siegel level and slope, drifting
slope = -1.5 + np.linspace(0, 1.2, 40)
shape = (1 - np.exp(-maturities.to_numpy() / 2)) / (maturities.to_numpy() / 2)
yields = pd.DataFrame(level[:, None] + slope[:, None] * shape[None, :], index=dates, columns=maturities)
fig_surface = surface_figure(yields, label="yield, %", title="The yield curve over time").view_init(elev=25, azim=-125)
fig_surface

In [ ]:
fig_wireframe = surface_figure(yields, kind="wireframe", label="yield, %", title="As a wireframe")
fig_wireframe

In [ ]:
fig_contours = surface_figure(yields, kind="contourf", levels=12, clabel=True, label="yield, %", title="As contours")
fig_contours

In [ ]:
curves = [Line3d(pd.Series(maturities.to_numpy(), name="maturity"), pd.Series([dates[k]] * len(maturities), name="date"),
                 yields.iloc[k].reset_index(drop=True).rename("yield"), label=str(dates[k])) for k in (0, 13, 26, 39)]
fig_curves = Figure((Panel(tuple(curves), title="Four of the curves"),))
fig_curves

In [ ]:
names = ["m3", "m6", "y1", "y2", "y3", "y5", "y7", "y10", "y20", "y30"]
curve_data = Dataset.from_frame(yields.set_axis(names, axis=1).rename_axis("time"))
fig_series_surface = curve_data.plot(names, kind="surface", title="Series side by side, as a surface")
fig_series_surface

## Results

Results draw their own figures, on their windows' tabs; `figure_of(result, tab)` gets one to restyle.
- **An equation:** actual, fitted and residuals; and a **Q-Q plot** of the residuals against the normal.
- **Coefficient plots** (`coefficient_figure`, `errorbar`): estimates with their intervals, several models side by side.
- **Impulse responses:** a grid of responses with bands, and a **heatmap** of every response by horizon (`imshow`).
- **Historical decompositions:** each shock's contribution as stacked bars, the data less the baseline as a line; a
  panel VAR's, one per entity (`hd[entity]`), and a proxy SVAR's (its shock, the other shocks together) draw the same way.
- **Correlations** (`kind="correlation"`) and **any table of numbers** (`heatmap_figure`) as heatmaps.

In [ ]:
short = Equation("gc ~ gy").fit(data)
long = Equation("gc ~ gy + gi").fit(data)
fig_fit = short.plot()
fig_fit

In [ ]:
fig_qq = short.qq_plot()
fig_qq

In [ ]:
fig_coefficients = coefficient_figure({"short": short, "long": long}, level=0.9, title="Consumption growth")
fig_coefficients

In [ ]:
var = VAR(["gy", "gc", "gi"], lags=2).fit(data)
irf = var.irf(12, draws=0)
fig_irf_heatmap = irf.heatmap()
fig_irf_heatmap

In [ ]:
hd = var.historical_decomposition()  # a VAR's; panel VARs and proxy SVARs give the same Decomposition
fig_historical = hd.plot(["gy"])
fig_historical

In [ ]:
fig_correlation = data.plot(["gy", "gc", "gi", "gg"], kind="correlation", title="Correlations of growth")
fig_correlation

In [ ]:
import pandas as pd
from pymetrix.estimation.priors import Beta, Gamma
from pymetrix.models.templates import new_keynesian

nk = new_keynesian()
sim = new_keynesian(sigma=2).solve().simulate(120, nk.shock_sd, seed=5)  # data from a slightly different model
sim.index = pd.period_range("1990Q1", periods=120, freq="Q", name="time")
dv = nk.dsge_var(nk.measurement.apply(Dataset.from_frame(sim)), lags=1, lambda_="grid", lambdas=[0.1, 0.25, 0.5, 1, 2, 5],
                 estimate=["rho", "kappa"], priors={"rho": Beta(0.7, 0.1), "kappa": Gamma(0.15, 0.05)})
fig_dsge_var_lambda = dv.lambda_plot()  # a DSGE-VAR's marginal data density by lambda: the hump, its top lambda-hat
fig_dsge_var_lambda

In [ ]:
fig_table = heatmap_figure(long, view="Covariance", label="covariance")  # any table of a result, by its tab's name
fig_table

In [ ]:
# estimates, t statistics and p-values are in different units: scale="auto" colours each column on its own scale
fig_table_by_column = heatmap_figure(long, view="Summary", scale="auto")
fig_table_by_column

In [ ]:
fig_retitled = figure_of(short).with_options(title="Consumption growth", shade="nber")  # a result's figure, changed
fig_retitled

## Styling, in matplotlib's terms

Each call takes matplotlib's own keywords and is recorded on the figure, so it replays: `line`, `bars` and `band`
(`Line2D.set`, `Rectangle.set`), `set` (`Axes.set`), `ticks` (`tick_params`, and a format), `legend`, `annotate`,
`text`, `axhline`... and `call` for any other Axes method that styles a panel. `style_steps()` lists the calls;
`with_style_of` copies them to another figure; `set_house_style` starts every figure from them.

In [ ]:
fig_styled = (data.plot(["gy", "gc"], span=("2015Q1", None), title="Styled")
              .line("gc", color="C3", linestyle="--", linewidth=2)
              .set(ylabel="% a year", ylim=(-15, 25))
              .ticks(axis="y", format="{x:.0f}%")
              .legend(loc="upper left", frameon=False)
              .annotate("COVID", xy=("2020Q2", -10), xytext=(20, -15), textcoords="offset points",
                        arrowprops={"arrowstyle": "->"})
              .call("minorticks_on"))
fig_styled.style_steps()

In [ ]:
fig_styled

In [ ]:
fig_copied = data.plot(["gc", "gi"], span=("2015Q1", None)).with_style_of(fig_styled)  # where the calls apply
fig_copied.style_steps()

In [ ]:
set_house_style(fig_styled)  # every figure drawn from now on starts from these calls, where they apply
print(len(house_style()), "house style calls")
fig_house = data.plot(["gc", "gi"], span=("2015Q1", None), title="In the house style")
fig_house

In [ ]:
set_house_style(None)  # back to plain figures

## Out of pymetrix

`fig.save(path, size=...)` writes PNG, PDF, SVG or EPS at a journal's column or page width, a slide's, or any size,
with the fonts embedded and what made the file recorded in it. `fig.matplotlib_code()` writes the chart as a
standalone matplotlib script; print it to see the matplotlib behind any figure above.

In [ ]:
print(fig_bars.matplotlib_code())

## pymetrix used in this lab

Every pymetrix function, class and method this lab calls, with its arguments and the first line of its documentation. For the full documentation, type the name and a question mark in a cell (`backtest?`), or press Shift+Tab inside the brackets of a call.

### From `pymetrix`

| | |
|---|---|
| `coefficient_figure(results, terms, level, title)` | A coefficient plot: each estimate with its interval at `level`, from one result or several (a mapping of names to results, or a sequence), side by side for each coefficient. |
| `Equation(text, params, family, priors, name, errors, variance)` | An equation written as text: `"c ~ y + c[-1]"` to estimate, or `"y = c + i + g"` (an identity). |
| `Figure(panels, title, columns, width, height, subtitle, footnote, theme, span, shade, events, reference, log, ylim, sharey, bar_labels, line_labels, style, house, ybreak, xbreak)` | Panels of layers, laid out `columns` across; `width` in inches. |
| `figure_of(obj, tab)` | The figure on a result's `tab` (its Graph tab by default), to change with `with_options`, e.g. `figure_of(fit).with_options(title="Consumption", shade="2008Q3:2009Q2")`. |
| `heatmap_figure(values, title, label, view, scale, diagnostics, **options)` | A table as a heatmap (`Heatmap`): a DataFrame, a `Table` (its numeric columns), or a result's table named by `view` (`heatmap_figure(granger, view="Results")`). |
| `house_style()` | The house style's calls (empty: none). |
| `set_house_style(style, notes)` | Make a figure's recorded style the house style: every figure drawn from now on starts from it, call by call where the call applies (as `with_style_of`: its series, panels and right-hand axes), and its own calls come after, so they win. |
| `surface_figure(values, kind, title, label, xlabel, ylabel, **options)` | A table over a grid, its columns along x and its rows along y (numbers, dates or names), each value a height: `kind` surface or wireframe (3-D, `Surface`), contourf (filled contours) or contour (lines; `Contour`). |
| `VAR(variables, lags, max_lags, exog, trend, by, factors)` | Vector autoregression, estimated equation by equation by least squares. |

### From `pymetrix.estimation.priors`

| | |
|---|---|
| `Beta(mean, sd)` | A beta prior on (0, 1), given by its mean and standard deviation (as Dynare's beta_pdf). |
| `Gamma(mean, sd)` | A gamma prior on (0, inf), given by its mean and standard deviation (as Dynare's gamma_pdf). |

### From `pymetrix.figures`

| | |
|---|---|
| `Line3d(x, y, z, label, color, width)` | A line through the points (x, y, z) on a 3-D panel (`Axes3D.plot`). |
| `Panel(layers, title, ylabel, xlabel, legend, ylabel_right, zlabel, projection, legend_loc)` | One panel of a figure (a matplotlib Axes): its layers, title and axis labels. |
| `vintages_figure(vintages, title, ylabel)` | One line per release of a series, oldest first: the older ones lighter and thinner, the latest darkest, to show how the data were revised. |

### From `pymetrix.models.templates`

| | |
|---|---|
| `new_keynesian(sigma, beta, kappa, rho, phi_pi, phi_y, pi_ss, r_ss, growth, shock_sd, lower_bound)` | The three-equation New Keynesian model. |

### Methods and properties of the results

| | |
|---|---|
| `Dataset.example(name)` | An example dataset shipped with pymetrix (see `pymetrix.datasets.available()`). |
| `Dataset.from_frame(df, time, entity, freq, variables, meta, source)` | Build a Dataset from a DataFrame, naming its time and entity columns. |
| `Dataset.plot(names, kind, layout, span, entity, title, stacked, right, transform, shade, total, coverages, bins, **options)` | A figure of series. |
| `Dataset.with_series(within, **exprs)` | Add or replace series, from expressions or from aligned Series; `None` makes a series of missing values (EViews' `series y`). |
| `Decomposition.plot(variables)` | A panel for each variable: the contributions as stacked bars, the path being decomposed as a line. |
| `Equation.fit(data, span, where, method, weights, missing)` | Estimate the equation on `data`: by least squares unless `method` says otherwise (`OLS(cov="hac")`, `IV(...)`, `ML()`, `Ridge()`, ...). |
| `Figure.annotate(text, xy, xytext, ax, **props)` | A note at `xy`, as `Axes.annotate` takes it: `fig.annotate("Lehman", xy=("2008Q3", 3.2), xytext=("2008Q3", 6), arrowprops=dict(arrowstyle="->"))`. |
| `Figure.call(method, *args, ax, side, **kwargs)` | Any other Axes method that styles or annotates a panel, recorded as data: `fig.call("set_aspect", "auto")`, `fig.call("ticklabel_format", axis="y", style="sci", scilimits=(0, 0))`, `fig.call("minorticks_on")`. |
| `Figure.legend(ax, **props)` | The legend, as `Axes.legend` takes it: `loc`, `ncols`, `frameon`, `fontsize`, `title`, `bbox_to_anchor`...; `visible=False` removes it. |
| `Figure.line(label, ax, **props)` | Style the line of the series `label`, as `Line2D.set` takes it: `color`, `linestyle`, `linewidth`, `marker`, `markersize`, `alpha`, `zorder` and the rest (aliases such as `lw` and `ls` too). |
| `Figure.matplotlib_code()` | A standalone script (NumPy, pandas and matplotlib only) that draws this figure from its numbers, every option and style call written as the matplotlib call it is: to check the chart, or reuse it, without pymetrix. |
| `Figure.set(ax, side, **props)` | Set a panel's axes, as `Axes.set` takes it: `xlim`, `ylim` (dates as periods, e.g. ("2010Q1", "2020Q4")), `xscale`, `yscale`, `xlabel`, `ylabel`, `title`, `aspect`... |
| `Figure.style_steps()` | The recorded style calls, in the order applied, each as the call that records it: `line('gc', color='C3')`, `set(ax=0, ylabel='%')`, `annotate(text='Lehman', xy=('2008Q3', 3.2), ax=0)`. |
| `Figure.ticks(ax, axis, format, **props)` | Ticks, as `Axes.tick_params` takes them (`labelsize`, `rotation`, `direction`, `length`...), and `format`, a format string for the tick labels as `StrMethodFormatter` takes it: `"{x:,.0f}"`, `"{x:.0%}"`. |
| `Figure.view_init(elev, azim, roll, ax)` | Turn a 3-D panel, as `Axes3D.view_init` does: `elev`, the angle above the x-y plane, and `azim`, the angle around the vertical axis, in degrees (matplotlib's default view is elev=30, azim=-60); `roll` about the line of sight. |
| `Figure.with_options(**options)` | A copy drawn differently. |
| `Figure.with_style_of(other, notes)` | A copy styled as `other` is: its recorded style calls added after this figure's own, where they apply. |
| `FittedDSGEVAR.lambda_plot()` | The log marginal data density against lambda (the hump), lambda-hat marked. |
| `FittedEquation.plot()` | Actual, fitted and residuals (time series), or actual against fitted (cross-sections). |
| `FittedEquation.qq_plot()` | The residuals against the normal distribution (a Q-Q plot): they lie along the dashed line if normal. |
| `FittedVAR.historical_decomposition(draws, alpha, seed)` | Each variable over the estimation sample, split into the contributions of the identified structural shocks and the baseline: the path from the first observations and the deterministic and exogenous terms, with no shocks. |
| `FittedVAR.irf(periods, orthogonal, cumulative, draws, alpha, seed)` | Impulse responses to one-standard-deviation structural shocks (or unit reduced-form shocks with `orthogonal=False`). |
| `IRF.heatmap(horizons)` | Every response to every shock at once: a row per response and shock, a column per horizon (the selected horizons by default), coloured on a scale symmetric about zero. |
| `LinearSolution.simulate(periods, shock_sd, seed, levels, shock_corr)` | A stochastic simulation: shocks drawn with standard deviations `shock_sd` (and correlations `shock_corr`); levels or deviations. |
| `MeasurementMap.apply(data)` | The observed series implied by model paths (a `Solution` or any dataset of the model's variables), in the units of the data: evaluated exactly, not linearised. |
| `Template.dsge_var(data, lags, lambda_, lambdas, estimate, priors, span, measurement, **kw)` | A DSGE-VAR(lambda) with the template's measurement equations, shocks and shock sizes (a `FittedDSGEVAR`): `Model.dsge_var`; see `pymetrix.models.dsge_var`. |
| `Template.solve(order)` | The rational-expectations solution: first order (a `LinearSolution`), or second order with the template's shock sizes (a `SecondOrderSolution`: risk corrections, quadratic terms). |
| `VAR.fit(data, span, method, transform, instrument_lags)` | Estimate the VAR on `data` over `span`: by least squares, equation by equation, or with `method=Bayes(...)` and a prior (e.g. Minnesota). |